**Assignment Overview**

In this assignment, you will explore two fundamental aspects of modern NLP systems: fine-tuning large language models and understanding the attention mechanism that powers them.

**Learning Goals**

By the end of this assignment, you should be able to:

* Understand when and why to fine-tune language models
* Apply QLoRA for efficient adaptation of large models
* Analyze model outputs and limitations
* Explain and implement the attention mechanism
* Interpret attention patterns and their behavior

**Important Note**

* Do not modify or delete the task structure.
* Complete each task with:
   - Clean, well-organized code
   - Relevant visualizations
   - Clear insights and explanations
* Make sure to answer all required questions.
**Submission requirements:**
* Submit a **fully executed notebook** (all cells must run and outputs should be visible).
* There is no need to attach the training and test datasets as files, but you must present them as DataFrame tables within the notebook.


## Install Required Packages and Dataset

In [1]:
!pip install -q bitsandbytes trl
!pip install fastapi uvicorn openai
!pip install datasets
!pip install datasets openai tqdm
!pip install -U trl
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastapi]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.6 MB/s  0:00:00 eta 0:00:01


In [20]:
import os
import sys
from getpass import getpass
from openai import OpenAI
from pydantic import BaseModel,ConfigDict,Field
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline,
    logging,
)
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model,PeftModel
from trl import SFTTrainer
import torch
import json
from datasets import Dataset, load_dataset
import pandas as pd


import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity(logging.CRITICAL)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.5.1).


ModuleNotFoundError: Could not import module 'Seq2SeqTrainer'. Are this object's requirements defined correctly?

### Load API Keys, Secrets, Base URLs and Model Names

In [11]:
def get_secret(key: str) -> str:
    """Fetch a secret from Colab userdata, local .env, or interactive prompt."""
    if "google.colab" in sys.modules:
        from google.colab import userdata
        value = userdata.get(key)
    else:
        from dotenv import load_dotenv
        load_dotenv()
        value = os.environ.get(key)

    if value is None:
        value = getpass(f"'{key}' not found in environment or secrets.\nEnter '{key}' manually: ")

    if not value:
        raise ValueError(f"'{key}' is empty.")
    return value


API_KEY_NAME = "NEBIUS_API_KEY"
api_key = get_secret(API_KEY_NAME)

NEBIUS_BASE_URL_NAME = "NEBIUS_BASE_URL"
nebius_base_url = get_secret(NEBIUS_BASE_URL_NAME)

Teacher_MODEL_NAME = "Qwen/Qwen3-30B-A3B-Instruct-2507"

## **Part 1 — QLoRA Fine-Tuning for Level-Adaptive Question Answering (75 points)**

In this part, you will fine-tune pretrained language models to answer questions at different explanation levels:

* Child — simple and intuitive explanation
* Student — clear educational explanation with moderate technical detail
* Expert — precise, technical, and domain-specific explanation

You will use a subset of the Databricks Dolly 15K dataset.

In [4]:
from datasets import Dataset, load_dataset
import pandas as pd

dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:5000]")


Generating train split: 100%|██████████| 15011/15011 [00:00<00:00, 255623.98 examples/s]


In [5]:
def is_good_question(text):
    return any(q in text.lower() for q in [
        "why", "how", "what is", "explain"
    ])

filtered = [
    ex for ex in dataset
    if ex["category"] == "open_qa" and is_good_question(ex["instruction"])
]

In [6]:
len(filtered), filtered[0]

(580,
 {'instruction': 'Why can camels survive for long without water?',
  'context': '',
  'response': 'Camels use the fat in their humps to keep them filled with energy and hydration for long periods of time.',
  'category': 'open_qa'})

### Dataset Samples by Quartiles

In [7]:
filtered_df = pd.DataFrame(filtered).drop(columns=['context', 'category'])
filtered_df['response_len'] = filtered_df['response'].apply(len)
filtered_df['len_quartile'] = pd.qcut(filtered_df['response_len'], q=4, duplicates='drop')
sampled_df = filtered_df.groupby('len_quartile').sample(n=10, random_state=42).sort_values(by='len_quartile')

for _, row in sampled_df.iterrows():
    print(f"quartile:{row['len_quartile']}")
    print(f"Instruction: {row['instruction']}")
    print(f"Response length: {row['response_len']}")
    print(f"Response: {row['response']}")
    print("-" * 50)



quartile:(1.999, 104.75]
Instruction: What is React?
Response length: 91
Response: React is a JavaScript library that specializes in helping developers build user interfaces.
--------------------------------------------------
quartile:(1.999, 104.75]
Instruction: How many feet are in 1 mile?
Response length: 10
Response: 5,280 feet
--------------------------------------------------
quartile:(1.999, 104.75]
Instruction: What is Bart Simpson's best friend named?
Response length: 9
Response: Millhouse
--------------------------------------------------
quartile:(1.999, 104.75]
Instruction: What is the fastest way to travel between the United States and Croatia?
Response length: 86
Response: The fastest way to travel would be by airplane, which would be faster than a boat trip
--------------------------------------------------
quartile:(1.999, 104.75]
Instruction: What is the state capitol of Nevada?
Response length: 99
Response: The state capitol of Nevada is Carson City which was founded 

### Quartile inspection — findings

We grouped the 580 filtered rows into 4 length-based quartiles and sampled 10 rows from each.

**Q1 (≤105 chars):**

Most Q1 rows (6 of 10 in the sample) are trivia questions, single fact lookups. Examples:
- "How many feet are in 1 mile?" → "5,280 feet"
- "What is Bart Simpson's best friend named?" → "Millhouse"

There's no concept to simplify or elaborate. The level-adaptation task is undefined for these questions.

**Q2 (105–264 chars):**

Q2 answers start to look more usable - they include definitions, comparisons. These can be worked into different levels 

**Q3 (264–476 chars):** 

Q3 has concept questions with answers that can be worked well into all three levels. 

**Q4 (>476 chars):** 

Q4 answers has some outliers with answers that are very long and detailed (e.g Debt Snowball and Avalanche at 1462 chars). 
A teacher LLM might unfaithfully rewrite that as a child answer, while omitting  / hallucinating.

### **Proposed further filtering:** min cap at 20 chars , max cap at 1000 chars. 
Filtering by length is imperfect, doesn't elimnate trivia question with long answers,  but still it is very simple to implement.



In [10]:
def is_good_answer_len(text):
    return 20 <= len(text) <=1000

length_filtered = [ex for ex in filtered if is_good_answer_len(ex["response"])]

len(length_filtered), length_filtered[0]


(505,
 {'instruction': 'Why can camels survive for long without water?',
  'context': '',
  'response': 'Camels use the fat in their humps to keep them filled with energy and hydration for long periods of time.',
  'category': 'open_qa'})

In [11]:
filtered_df = pd.DataFrame(length_filtered).drop(columns=['context', 'category'])
filtered_df['response_len'] = filtered_df['response'].apply(len)
filtered_df['len_quartile'] = pd.qcut(filtered_df['response_len'], q=4, duplicates='drop')
sampled_df = filtered_df.groupby('len_quartile').sample(n=10, random_state=42).sort_values(by='len_quartile')

for _, row in sampled_df.iterrows():
    print(f"quartile:{row['len_quartile']}")
    print(f"Instruction: {row['instruction']}")
    print(f"Response length: {row['response_len']}")
    print(f"Response: {row['response']}")
    print("-" * 50)


quartile:(19.999, 121.0]
Instruction: What is a Fixed asset in finance?
Response length: 116
Response: A fixed asset is one which is intended to be used for several years. Examples are buildings,
machinery and vehicles.
--------------------------------------------------
quartile:(19.999, 121.0]
Instruction: How many James Bond movies did Daniel Craig star in?
Response length: 43
Response: Daniel Craig starred in 5 James Bond movies
--------------------------------------------------
quartile:(19.999, 121.0]
Instruction: How many planets are there in the solar system?
Response length: 40
Response: There are 8 planets in the solar system.
--------------------------------------------------
quartile:(19.999, 121.0]
Instruction: What is a good dad joke?
Response length: 50
Response: What do you call an okay factory?  A satisfactory.
--------------------------------------------------
quartile:(19.999, 121.0]
Instruction: What is heavier? A pound of feathers or a pound of iron?
Response length

### Test Nebius API for Schema-enforced output 

In [19]:
client = OpenAI(base_url=nebius_base_url, api_key=api_key)

#Pydantic model to validate the response
class AnimalColor(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: str
    color: str

print("AnimalColor.model_json_schema():",AnimalColor.model_json_schema())


def test_helper(test_name, response_format=None, pydantic_model=None):
    kwargs = {"response_format": response_format} if response_format is not None else {}
    print(f"--- {test_name} ---")
    
    try:
        if pydantic_model:
            response = client.beta.chat.completions.parse(
                model=Teacher_MODEL_NAME,
                temperature=0.0,
                max_tokens=200,
                messages=[{"role": "user", "content": "Invent a fictional animal. Return a JSON object with two fields: name (a made-up animal name) and color (its color). Do not include any other text."}],
                response_format=pydantic_model)
            validated = response.choices[0].message.parsed 
        else:
            response = client.chat.completions.create(
                model=Teacher_MODEL_NAME,
                temperature=0.0,
                max_tokens=200,
                messages=[{"role": "user", "content": "Invent a fictional animal. Return a JSON object with two fields: name (a made-up animal name) and color (its color). Do not include any other text."}],
                **kwargs)
            validated = AnimalColor.model_validate_json(response.choices[0].message.content)

        print("Validated response:", validated)
    except Exception as e:
        print("\tError:", str(e))
        return e.__class__.__name__
    print("\tSuccess")
    return True


test_helper("Test A - plain , no constraints", response_format=None)
test_helper("Test B -  json_object mode", response_format={"type": "json_object"})
test_helper("Test C - json_schema", response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "AnimalColor",
        "strict": True,
        "schema": AnimalColor.model_json_schema()
    }})
test_helper("Test D - pydantic model", pydantic_model=AnimalColor)





AnimalColor.model_json_schema(): {'additionalProperties': False, 'properties': {'name': {'title': 'Name', 'type': 'string'}, 'color': {'title': 'Color', 'type': 'string'}}, 'required': ['name', 'color'], 'title': 'AnimalColor', 'type': 'object'}
--- Test A - plain , no constraints ---
Validated response: name='Luminox' color='pearl silver'
	Success
--- Test B -  json_object mode ---
Validated response: name='Luminox' color='pearl silver'
	Success
--- Test C - json_schema ---
Validated response: name='Luminox' color='pearl silver'
	Success
--- Test D - pydantic model ---
Validated response: name='Luminox' color='pearl silver'
	Success


True

### **Task 1.1 — Prepare a Custom Dataset (15 points)**

Create your own instruction-tuning dataset based on a subset of databricks-dolly-15k.

For each selected question-answer pair, create versions of the answer adapted to the requested expertise level.

Example format:

    Question: What is gradient descent?
    Expertise level: child
    Answer: Gradient descent is like walking downhill step by step until you reach the lowest point.

You should prepare:

1. Training set for fine-tuning -

   Use the filtered Dolly dataset as your source of questions.

   For each question, create examples in the example format per level.

2. Test set for evaluation -
    
   Create a small test set of 20 question-answer examples.

   The test set must:

  * include examples from all three expertise levels
  * be separate from the training set
  * be manually reviewed by you for quality
  * include answers that are appropriate for the requested expertise level

  Recommendation:

  1. Save the generated dataset as a '.jsonl' file so it can be loaded and reused later.

  2. Decide on the training format according to the model architecture:
    
    - For causal language models, such as `HuggingFaceTB/SmolLM2-360M-Instruct`, use one full text field:

    ```text
    ### Question:
    ...

    ### Expertise level:
    child / student / expert

    ### Answer:
    ...```

    - For seq2seq models, such as google/flan-t5-small, separate the input and target:

    input: Question + expertise level
    target: Answer
   
  3. Before generating the full dataset, test the pipeline on a small batch of examples to verify that:

  * the LLM returns valid JSON,
  * each question receives three expertise-level answers,
  * the saved .jsonl file can be loaded correctly,
  * the format matches the training code.

**Pydantic Schema and Prompt **

In [ ]:
# Pydantic model for leveled answers
class LeveledAnswers(BaseModel):
    """Three answers to the same question, calibrated to a child, a student, and an expert."""
    model_config = ConfigDict(extra="forbid")
    child: str = Field(description="Answer suitable for a child — simple and intuitive explanation, no jargon.")
    student: str = Field(description="Answer suitable for a student — clear educational explanation with moderate technical detail.")
    expert: str = Field(description="Answer suitable for an expert — precise, technical, and domain-specific explanation.")

print(LeveledAnswers.model_json_schema())


# Prompts for generating leveled answers
SYSTEM_PROMPT

{'additionalProperties': False, 'description': 'Three answers to the same question, calibrated to a child, a student, and an expert.', 'properties': {'child': {'description': 'Answer suitable for a child — simple and intuitive explanation, no jargon.', 'title': 'Child', 'type': 'string'}, 'student': {'description': 'Answer suitable for a student — clear educational explanation with moderate technical detail.', 'title': 'Student', 'type': 'string'}, 'expert': {'description': 'Answer suitable for an expert — precise, technical, and domain-specific explanation.', 'title': 'Expert', 'type': 'string'}}, 'required': ['child', 'student', 'expert'], 'title': 'LeveledAnswers', 'type': 'object'}


### **Task 1.2 — Fine-Tune Models with QLoRA (10 points)**

Fine-tune the models using QLoRA, meaning:

1. Load the base model in 4-bit quantization
2. Add LoRA adapters
3. Train only the adapter parameters

You should repeat the fine-tuning process for both models:

1. HuggingFaceTB/SmolLM2-360M-Instruct
2. google/flan-t5-small

Pay attention each model requires the dataset to be formated diffrently.

### **Task 1.3 — Evaluation (20 points)**

Evaluate both fine-tuned models on your 20-example test set.

For each model, compare:

* outputs before fine-tuning
* outputs after fine-tuning
* whether the answer matches the requested expertise level
* whether the answer is clear and relevant
* whether the answer stays faithful to the question

Evaluate using this table:

|Question|Level|Base Output|Fine-tuned Output|Expected Answer|Better after FT?|Notes|
|-------|-------|-------|-------|-------|-------|-------|


### **Task 1.4 — Model Comparison and Discussion (15 points)**

Compare the performance of the two models:

* Which model adapted better to the expertise levels?
* Which model produced clearer answers?
* Which model followed the requested format better?
* Did one model hallucinate more than the other?

Explain any differences you observe.

In your discussion, consider that:

* SmolLM2-360M-Instruct is a decoder-only instruction model
* flan-t5-small is an encoder-decoder instruction model
* Different architectures may behave differently on instruction-following and text generation tasks

### **Task 1.5 - Conceptual Questions (15 points)**

Answer the following questions:

1. What changed after fine-tuning?
   
   Discuss whether the model became better at adapting its answer to the requested expertise level.
2. Why is QLoRA more memory efficient?
   
   Explain the role of 4-bit quantization and LoRA adapters.
3. What happens if you increase the LoRA rank?
   
   Discuss the tradeoff between model capacity, memory usage, and overfitting risk.
4. Why use LoRA / QLoRA instead of prompt engineering?

      Discuss:

      * In what cases prompt engineering is sufficient
      * When fine-tuning becomes necessary
      * What advantages QLoRA provides over prompting
      * What are the trade-offs (cost, flexibility, control)

## **Part 2 — Understanding Attention (25 points)**

Goal:

Build intuition for how attention works.

Task:

You will implement a simple attention mechanism from scratch (PyTorch) and visualize its behavior.

### **Task 2.1 Implement Scaled Dot-Product Attention (5 points)**

Given:

* Query (Q)
* Key (K)
* Value (V)

Compute:

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Implements Scaled Dot-Product Attention.

    Args:
        Q: (batch, heads, seq_len_q, d_k)
        K: (batch, heads, seq_len_k, d_k)
        V: (batch, heads, seq_len_v, d_v)
        mask: (batch, heads, seq_len_q, seq_len_k) or None

    Returns:
        output: (batch, heads, seq_len_q, d_v)
        attention_weights: (batch, heads, seq_len_q, seq_len_k)
    """


    #  Get dimension for scaling
    d_k = Q.size(-1)


    # Compute attention scores

    scores = # TODO: compute dot-product between Q and K^T


    # Scale the scores

    # TODO: divide scores by sqrt(d_k)


    #  Apply mask (if given)

    if mask is not None:
        # TODO: mask out invalid positions (set to -inf)
        pass


    # Softmax to get attention

    attention_weights = # TODO: apply softmax over last dimension


    # Compute weighted sum

    output = #TODO: multiply attention_weights with V

    return output, attention_weights

### **Task 2.2 Visualize Attention (5 points)**


Plot attention weights as a heatmap for the given sentence


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import math



sentence = "the cat sat on the mat"


tokens = # TODO: split sentence into tokens

seq_len = # TODO: compute sequence length



# Create dummy embeddings

d_model = 16


embeddings = # TODO: initialize random embeddings of shape (seq_len, d_model)

# Treat embeddings as Q, K, V
Q = embeddings
K = embeddings
V = embeddings



# Compute Scaled Dot-Product Attention

attention_weights = # TODO: compute attention weights (use function from above)



# Plot attention heatmap

plt.figure(figsize=(8, 6))

# TODO: plot heatmap using seaborn
# - use attention_weights
# - set xticklabels and yticklabels to tokens
# - choose a colormap (e.g., "viridis")

plt.title("Attention Weights Heatmap")

# TODO: label axes
# plt.xlabel(...)
# plt.ylabel(...)

# TODO: rotate ticks if needed

plt.show()



### **Task 2.3 Experiments (15 points)**

**Experiment 1 — Change One Word:**

1. Use a simple sentence, for example:
    the cat sat on the mat
2. Compute and plot the attention weights as a heatmap.
3. Change one word in the sentence, for example:
    the dog sat on the mat
4. Recompute and plot the attention heatmap.
5. Compare the two heatmaps and explain whether the attention pattern changed.

**Experiment 2 — Compare Different Attention Heads:**

Repeat the attention visualization using at least two different attention heads.

For each head, create separate projection matrices:

$$W_Q, W_K, W_V$$

Use them to compute:
$$Q=XW_Q, K=XW_K, V=XW_V$$

Then compute and plot the attention weights for each head.

**Questions**

Answer briefly:

1. Did changing one word affect the attention weights? Why or why not?
2. Do different attention heads focus on different tokens?
3. Why might multi-head attention be useful in Transformer models?